In [ ]:
import duckdb
import os
from tqdm.notebook import tqdm
import pandas as pd

duckdb.execute('INSTALL sqlite')

db_dir = '../../data/'

column_name_counts = {}

table_name = 'video_statistics'

for db in tqdm(os.listdir(db_dir)):
    if db.endswith('.db'):
        print(db)
        conn = duckdb.connect(db_dir + '/' + db)
        # print list of tables
        tables = conn.execute("SELECT name FROM sqlite_master WHERE type='table';").fetchall()
        tables = [t[0] for t in tables]
        if table_name in tables:
            # print schema
            schema = conn.execute(f"PRAGMA table_info({table_name});").fetchall()
            column_names = [s[1] for s in schema]
            # if 'video_id' in column_names:
            #     print(f'{db} has video_id')
            for col in column_names:
                if col not in column_name_counts:
                    column_name_counts[col] = 1
                else:
                    column_name_counts[col] += 1
        else:
            print('No search_results table', db)
        conn.close()

print()
for col, count in column_name_counts.items():
    print(f'{col}: {count}')

In [ ]:
# Join search_results tables

import pandas as pd

import duckdb
import os

db_dir = '../../data/'
dbs = [filename for filename in os.listdir(db_dir) if '.db' in filename]

# Step 1: Get all unique columns from all databases
all_columns = set()

datasets = []

rename_map = {
    'id': 'id.videoId',
}

for db in dbs:
    
    conn = duckdb.connect(db_dir + '/' + db)
    
    tables = conn.query("SELECT name FROM sqlite_master WHERE type='table';").fetchdf()['name'].values

    if 'search_results' in tables:

        data = conn.query("SELECT * FROM search_results").df()
        
        data = data.rename(columns=rename_map)
        
        data.drop(columns=['kind', 'id.kind', 'snippet.publishTime'], inplace=True, errors='ignore')
        
        data['id.kind'] = 'youtube#video'
        
        datasets.append(data)

    conn.close()

search_results_df = pd.concat(datasets)

search_results_df.drop_duplicates(subset=['id.videoId'], inplace=True)

search_results_df.set_index('id.videoId', inplace=True)

conversion_columns = ['snippet.thumbnails.default.width', 'snippet.thumbnails.default.height', 'snippet.thumbnails.medium.width', 
                      'snippet.thumbnails.medium.height', 'snippet.thumbnails.high.width', 'snippet.thumbnails.high.height', 
                      'snippet.thumbnails.standard.width', 'snippet.thumbnails.standard.height', 'snippet.thumbnails.maxres.width',
                      'snippet.thumbnails.maxres.height', 'contentDetails.licensedContent', 'statistics.viewCount', 
                      'statistics.likeCount', 'statistics.favoriteCount', 'statistics.commentCount']

for col in conversion_columns:
    if col in search_results_df.columns:
        search_results_df[col] = pd.to_numeric(search_results_df[col], errors='coerce')
        search_results_df[col].fillna(0, inplace=True)
        search_results_df[col] = search_results_df[col].astype(int)

# Now try saving to parquet again
search_results_df.to_parquet('../../data/joins/search_results.parquet')

In [ ]:
# Join channel_statistics tables

import duckdb
import os

db_dir = '../../data/'
dbs = [filename for filename in os.listdir(db_dir) if '.db' in filename]


# Step 1: Get all unique columns from all databases
all_columns = set()

datasets = []

for db in dbs:
    conn = duckdb.connect(db_dir + '/' + db)
    
    tables = conn.query("SELECT name FROM sqlite_master WHERE type='table';").fetchdf()['name'].values

    if 'channel_statistics' in tables:

        data = conn.query("SELECT * FROM channel_statistics").df()
        
        datasets.append(data)

    conn.close()

import pandas as pd

channel_statistics_df = pd.concat(datasets)

channel_statistics_df.drop_duplicates(subset=['id'], inplace=True)

channel_statistics_df.set_index('id', inplace=True)


conversion_columns = ['snippet.thumbnails.default.width', 'snippet.thumbnails.default.height', 'snippet.thumbnails.medium.width', 
                      'snippet.thumbnails.medium.height', 'snippet.thumbnails.high.width', 'snippet.thumbnails.high.height', 
                      'snippet.thumbnails.standard.width', 'snippet.thumbnails.standard.height', 'snippet.thumbnails.maxres.width',
                      'snippet.thumbnails.maxres.height', 'contentDetails.licensedContent', 'statistics.viewCount', 
                      'statistics.likeCount', 'statistics.favoriteCount', 'statistics.commentCount']

for col in conversion_columns:
    if col in channel_statistics_df.columns:
        channel_statistics_df[col] = pd.to_numeric(channel_statistics_df[col], errors='coerce')
        channel_statistics_df[col].fillna(0, inplace=True)
        channel_statistics_df[col] = channel_statistics_df[col].astype(int)

# Now try saving to parquet again
channel_statistics_df.to_parquet('../../data/joins/channel_statistics.parquet')

In [ ]:
# Join video_statistics tables

import duckdb
import os

db_dir = '../../data/'
dbs = [filename for filename in os.listdir(db_dir) if '.db' in filename]

# Step 1: Get all unique columns from all databases
all_columns = set()

rename_map = {
    'video_id': 'id',
    'video_title': 'snippet.title',
    'video_published_at': 'snippet.publishedAt',
    'channel_id': 'snippet.channelId',
    'video_description': 'snippet.description',
    'channel_title': 'snippet.channelTitle',
    'video_tags': 'snippet.tags',
    'video_duration': 'contentDetails.duration',
    'video_caption': 'contentDetails.caption',
    'video_licensed_content': 'contentDetails.licensedContent',
    'video_view_count': 'statistics.viewCount',
    'video_like_count': 'statistics.likeCount',
    'video_comment_count': 'statistics.commentCount',
    'video_topic_categories': 'topicDetails.topicCategories',
}

datasets = []

from tqdm.notebook import tqdm

for db in tqdm(dbs):
    
    print(db)
    
    conn = duckdb.connect(db_dir + '/' + db)
    
    tables = conn.query("SELECT name FROM sqlite_master WHERE type='table';").fetchdf()['name'].values

    if 'video_statistics' in tables:

        data = conn.query("SELECT * FROM video_statistics").df()
        
        data = data.rename(columns=rename_map)
       
        data.drop(columns=['index'], inplace=True, errors='ignore')
        
        datasets.append(data)

    conn.close()

import pandas as pd

video_statistics_df = pd.read_parquet('../../data/joins/video_statistics.parquet')

# video_statistics_df = pd.concat(datasets)

# video_statistics_df.drop_duplicates(subset=['id'], inplace=True)

# video_statistics_df.set_index('id', inplace=True)

# video_statistics_df.shape

video_statistics_df

In [8]:
import polars as pl

video_statistics_df = pl.read_parquet('../../data/joins/video_statistics.parquet')

video_statistics_df

ComputeError: parquet: File out of specification: underlying IO error: Allocation error : not enough memory

In [ ]:
import duckdb

data_loc = '../../data/db/data.db'

conn = duckdb.connect(data_loc)

db_data = conn.query("SELECT * FROM video_statistics").pl()

db_data.shape

In [ ]:
import polars as pl

existing_video_statistics = pl.read_parquet('../../data/joins/video_statistics.parquet')
existing_video_statistics.shape

In [ ]:
import polars as pl

# Ensure both dataframes have the same columns
db_data = db_data.select(existing_video_statistics.columns)

# Align the data types of `existing_video_statistics` to match `data`
for col in existing_video_statistics.columns:
    if existing_video_statistics[col].dtype != db_data[col].dtype:
        db_data = db_data.with_columns(
            db_data[col].cast(existing_video_statistics[col].dtype).alias(col)
        )

# Concatenate the two dataframes
video_statistics_df = pl.concat([existing_video_statistics, db_data])

video_statistics_df = video_statistics_df.unique('id')

# Get the shape of the resulting dataframe
video_statistics_df.shape

In [ ]:
video_statistics_df.write_parquet('../../data/joins/video_statistics.parquet')
video_statistics_df = pl.read_parquet('../../data/joins/video_statistics.parquet')
video_statistics_df

In [ ]:
import duckdb
import os

db_dir = '../data/db'
dbs = os.listdir(db_dir)

# Step 1: Get all unique columns from all databases
all_columns = set()

has_channel_statistics = []
for db in dbs:
    conn = duckdb.connect(db_dir + '/' + db)
    tables = conn.execute("SELECT name FROM sqlite_master WHERE type='table';").fetchdf()
    if 'channel_statistics' in tables['name'].tolist():
        query = f"SELECT * FROM sqlite_scan('{db_dir}/{db}', 'channel_statistics') LIMIT 1"
        result = conn.execute(query).fetchdf()
        all_columns.update(result.columns.tolist())
    conn.close()

# Convert the set to a list to maintain order
all_columns = list(all_columns)

# Step 2: Build the queries
queries = []
for db in dbs:
    conn = duckdb.connect(db_dir + '/' + db)
    tables = conn.execute("SELECT name FROM sqlite_master WHERE type='table';").fetchdf()
    if 'channel_statistics' in tables['name'].tolist():
        # Fetch the actual columns in this database
        query = f"SELECT * FROM sqlite_scan('{db_dir}/{db}', 'channel_statistics') LIMIT 1"
        result = conn.execute(query).fetchdf()
        db_columns = result.columns.tolist()
        # Create a list of column selections, using NULL for missing columns
        column_selections = []
        for col in all_columns:
            if col in db_columns:
                # Escape column names with periods using double quotes
                column_selections.append(f'"{col}"')
            else:
                # Handle missing columns
                column_selections.append(f"NULL as \"{col}\"")
        # Build the full query for this database
        queries.append(f"SELECT {', '.join(column_selections)} FROM sqlite_scan('{db_dir}/{db}', 'channel_statistics')")
    conn.close()
    
# Step 3: Combine all queries with UNION ALL
joined_query = ' UNION '.join(queries)

conn = duckdb.connect(':memory:')

data = conn.query(joined_query)

# Step 1: Connect to or create the new database
new_conn = duckdb.connect('total.db')

new_conn.execute(f"DROP TABLE IF EXISTS channel_statistics")

# Step 2: Write the query result into the 'search_results' table in the new database
new_conn.execute(f"CREATE TABLE channel_statistics AS {joined_query}")

# Optionally: Verify the results by querying the new table
result = new_conn.execute("SELECT * FROM channel_statistics").fetchdf()

print(result)

In [ ]:
# Optionally: Verify the results by querying the new table
result = new_conn.execute("SELECT * FROM search_results").fetchdf()
print(result)

In [ ]:
import os

from sqlalchemy import create_engine, inspect

from tqdm.notebook import tqdm

db_dir = '../data/db'

os.listdir(db_dir)

table_names = set()
for db in tqdm(os.listdir(db_dir)):
    engine = create_engine(f'sqlite:///{db_dir}/{db}', echo=False)
    inspector = inspect(engine)
    table_names.update(inspector.get_table_names())

table_names

In [ ]:
import os

import pandas as pd

from sqlalchemy import create_engine, inspect

from tqdm.notebook import tqdm

db_dir = '../data/db'

os.listdir(db_dir)

has_search_results_dbs = []
for db in tqdm(os.listdir(db_dir)):
    engine = create_engine(f'sqlite:///{db_dir}/{db}', echo=False)
    inspector = inspect(engine)
    print(db) 
    print(inspector.get_table_names())
    print()

In [ ]:
import os

import pandas as pd

from sqlalchemy import create_engine, inspect

from tqdm.notebook import tqdm

db_dir = '../data/db'

os.listdir(db_dir)

has_search_results_dbs = []
for db in tqdm(os.listdir(db_dir)):
    engine = create_engine(f'sqlite:///{db_dir}/{db}', echo=False)
    inspector = inspect(engine)
    if inspector.has_table('search_results'):
        has_search_results_dbs.append(db)
        
search_results_tables = []
for db in tqdm(has_search_results_dbs):
    engine = create_engine(f'sqlite:///{db_dir}/{db}', echo=False)
    search_results_tables.append(pd.read_sql_table('search_results', engine))
    
search_results_df = pd.concat(search_results_tables)
search_results_df = search_results_df[search_results_df['statistics.viewCount'].astype(str).str.isdigit()]
search_results_df['statistics.viewCount'] = search_results_df['statistics.viewCount'].astype(int)
search_results_df.sort_values('statistics.viewCount', ascending=False, inplace=True)
search_results_df.drop_duplicates(subset='id', keep='first', inplace=True)
search_results_df.set_index('id', inplace=True)
search_results_df.dropna(axis=1, how='all', inplace=True)
search_results_df

In [ ]:
# create new db '../data/data.db'
engine = create_engine(f'sqlite:///../data/data.db', echo=False)

# write search_results_df to data.db as table search_results
search_results_df.to_sql('search_results', engine, if_exists='replace', index=True)

In [ ]:
# read search_results table
search_results_df = pd.read_sql_table('search_results', engine)

search_results_df

In [ ]:
import os

import pandas as pd

from sqlalchemy import create_engine, inspect

from tqdm.notebook import tqdm

db_dir = '../data/db'

os.listdir(db_dir)

has_search_results_dbs = []
for db in tqdm(os.listdir(db_dir)):
    engine = create_engine(f'sqlite:///{db_dir}/{db}', echo=False)
    inspector = inspect(engine)
    if inspector.has_table('search_results'):
        has_search_results_dbs.append(db)
        
search_results_tables = []
for db in tqdm(has_search_results_dbs):
    engine = create_engine(f'sqlite:///{db_dir}/{db}', echo=False)
    search_results_tables.append(pd.read_sql_table('search_results', engine))
    
search_results_df = pd.concat(search_results_tables)
search_results_df = search_results_df[search_results_df['statistics.viewCount'].astype(str).str.isdigit()]
search_results_df['statistics.viewCount'] = search_results_df['statistics.viewCount'].astype(int)
search_results_df.sort_values('statistics.viewCount', ascending=False, inplace=True)
search_results_df.drop_duplicates(subset='id', keep='first', inplace=True)
search_results_df.set_index('id', inplace=True)
search_results_df.dropna(axis=1, how='all', inplace=True)
search_results_df

In [ ]:
import os

import pandas as pd

from sqlalchemy import create_engine, inspect

from tqdm.notebook import tqdm

db_dir = '../data/db'

os.listdir(db_dir)

has_channel_statistics_dbs = []
for db in tqdm(os.listdir(db_dir)):
    engine = create_engine(f'sqlite:///{db_dir}/{db}', echo=False)
    inspector = inspect(engine)
    if inspector.has_table('channel_statistics'):
        has_channel_statistics_dbs.append(db)
        
channel_statistics_tables = []
for db in tqdm(has_channel_statistics_dbs):
    engine = create_engine(f'sqlite:///{db_dir}/{db}', echo=False)
    channel_statistics_tables.append(pd.read_sql_table('channel_statistics', engine))

channel_statistics_df = pd.concat(channel_statistics_tables)
channel_statistics_df.drop_duplicates(subset='id', inplace=True)
channel_statistics_df.set_index('id', inplace=True)
channel_statistics_df.dropna(axis=1, how='all', inplace=True)

drop_cols = ['channel_id', 'channel_title', 'channel_description', 'channel_published_at',
             'channel_view_count', 'channel_subscriber_count', 'channel_video_count',
             'channel_topic_categories']

channel_statistics_df.drop(columns=drop_cols, inplace=True)

channel_statistics_df

In [ ]:
import os

import pandas as pd

from sqlalchemy import create_engine, inspect

from tqdm.notebook import tqdm

db_dir = '../data/db'

os.listdir(db_dir)

has_channel_playlists_dbs = []
for db in tqdm(os.listdir(db_dir)):
    engine = create_engine(f'sqlite:///{db_dir}/{db}', echo=False)
    inspector = inspect(engine)
    if inspector.has_table('channel_playlists'):
        has_channel_playlists_dbs.append(db)        

channel_playlists_tables = []
pbar = tqdm(has_channel_playlists_dbs)
for db in has_channel_playlists_dbs:
    pbar.set_description(f'Processing {db}')
    engine = create_engine(f'sqlite:///{db_dir}/{db}', echo=False)
    channel_playlists_tables.append(pd.read_sql_table('channel_playlists', engine))
    pbar.update(1)
    
channel_playlists_df = pd.concat(channel_playlists_tables)
channel_playlists_df
